In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

display(HTML("<style>.output_scroll {height: auto !important; max-height: none !important;}</style>"))

display(HTML("""
<script>
    if (window.MathJax) {
        MathJax.Hub.Config({
            "HTML-CSS": { scale: 130 },
            SVG: { scale: 130 }
        });
    }
</script>
<style>
    .rendered_html math { font-size: 1.3em !important; }
</style>
"""))

sp.init_printing(use_latex='mathjax')

z, n, k, N = sp.symbols('z n k N', complex=True, integer=True)
H = sp.symbols('H')

equation = sp.Eq(H*z**n - sp.Rational(1, 4)*H*z**(n - 1), z**n)
H_z = sp.factor(sp.solve(equation, H)[0])

omega = sp.symbols('omega', real=True)
H_ejw = sp.simplify(H_z.subs(z, sp.exp(sp.I*omega)))

def dtfs_coefficients(x, N_value):
    m = sp.Symbol('m', integer=True)
    coeffs = []
    for r in range(N_value):
        term = x.subs(n, m) * sp.exp(-sp.I * 2 * sp.pi * r * m / N_value)
        summation = sp.Sum(term, (m, 0, N_value - 1)).doit()
        val = sp.Rational(1, N_value) * summation
        coeffs.append(sp.simplify(sp.expand_complex(val)))
    return coeffs


def output_coefficients(alpha, N_value):
    beta = []
    for r in range(N_value):
        omega_val = sp.Rational(2 * r, N_value) * sp.pi
        H_r = sp.simplify(H_ejw.subs(omega, omega_val))
        beta_r = alpha[r] * H_r
        beta.append(sp.simplify(sp.expand_complex(beta_r)))
    return beta


def fourier_synthesis(coefficients, N_value):
    synthesis_sum = sum(
        coefficients[r] * sp.exp(sp.I * 2 * sp.pi * r * n / N_value)
        for r in range(N_value)
    )
    return sp.simplify(synthesis_sum)


def real_fourier_form(coefficients, N_value):
    result = 0
    for r in range(N_value):
        if r == 0:
            result += coefficients[r]
        elif r == N_value // 2:
            result += coefficients[r] * sp.exp(sp.I * sp.pi * n)
        else:
            c = coefficients[r]
            a = sp.simplify(sp.re(c))
            b = sp.simplify(sp.im(c))
            omega_r = sp.Rational(2 * r, N_value) * sp.pi
            result += 2 * (a * sp.cos(omega_r * n) - b * sp.sin(omega_r * n))
    return sp.simplify(result)

def display_nonzero_coefficients(coefficients, name, N_value):
    display(HTML(f"<b>Non-zero Fourier coefficients of {name}:</b>"))
    for r, coefficient in enumerate(coefficients):
        if coefficient != 0:
            idx = r if r <= N_value // 2 else r - N_value
            display(sp.Eq(sp.Symbol(f'{name}_{{{idx}}}'), coefficient))

x1, N1 = sp.sin(3*sp.pi*n/4), 8
display(HTML("<b>Input signal:</b>"))
display(sp.Eq(sp.Symbol('x_1[n]'), x1))

alpha1 = dtfs_coefficients(x1, N1)
display_nonzero_coefficients(alpha1, r'\alpha', N1)

beta1 = output_coefficients(alpha1, N1)
display(HTML("<b>Non-zero output Fourier coefficients:</b>"))
for r, coefficient in enumerate(beta1):
    if coefficient != 0:
        idx = r if r <= N1 // 2 else r - N1
        display(sp.Eq(sp.Symbol(f'y_{{{idx}}}'), coefficient))

y1_complex, y1_real = fourier_synthesis(beta1, N1), real_fourier_form(beta1, N1)
display(HTML("<b>Complex Fourier expansion of the output:</b>"))
display(sp.Eq(sp.Symbol('y_1[n]'), y1_complex))
display(HTML("<b>Real-valued Fourier expansion of the output:</b>"))
display(sp.Eq(sp.Symbol('y_1[n]'), y1_real))

verification1 = sp.simplify(y1_real - sp.Rational(1, 4)*y1_real.subs(n, n - 1) - x1)
display(HTML("<b>Verification using the original difference equation:</b>"))
display(sp.Eq(sp.Symbol('y_1[n]') - sp.Rational(1,4)*sp.Symbol('y_1[n-1]') - sp.Symbol('x_1[n]'), verification1))

x2, N2 = sp.cos(sp.pi*n/4) + 2*sp.cos(sp.pi*n/2), 8
display(HTML("<b>Input signal:</b>"))
display(sp.Eq(sp.Symbol('x_2[n]'), x2))

alpha2 = dtfs_coefficients(x2, N2)
display_nonzero_coefficients(alpha2, r'\alpha', N2)

beta2 = output_coefficients(alpha2, N2)
display(HTML("<b>Non-zero output Fourier coefficients:</b>"))
for r, coefficient in enumerate(beta2):
    if coefficient != 0:
        idx = r if r <= N2 // 2 else r - N2
        display(sp.Eq(sp.Symbol(f'y_{{{idx}}}'), coefficient))

y2_complex, y2_real = fourier_synthesis(beta2, N2), real_fourier_form(beta2, N2)
display(HTML("<b>Complex Fourier expansion of the output:</b>"))
display(sp.Eq(sp.Symbol('y_2[n]'), y2_complex))
display(HTML("<b>Real-valued Fourier expansion of the output:</b>"))
display(sp.Eq(sp.Symbol('y_2[n]'), y2_real))

verification2 = sp.simplify(y2_real - sp.Rational(1, 4)*y2_real.subs(n, n - 1) - x2)
display(HTML("<b>Verification using the original difference equation:</b>"))
display(sp.Eq(sp.Symbol('y_2[n]') - sp.Rational(1,4)*sp.Symbol('y_2[n-1]') - sp.Symbol('x_2[n]'), verification2))

display(HTML("<h3>Final result for x₁[n]</h3>"))
display(sp.Eq(sp.Symbol('y_1[n]'), y1_real))
display(HTML("<h3>Final result for x₂[n]</h3>"))
display(sp.Eq(sp.Symbol('y_2[n]'), y2_real))

def plot_signal_pair(x, y, N_value, title):
    samples = np.arange(0, 2*N_value)
    x_values = [float(sp.N(x.subs(n, int(r)))) for r in samples]
    y_values = [float(sp.N(y.subs(n, int(r)))) for r in samples]
    fig, ax = plt.subplots(2, 1, figsize=(10, 5))
    ax[0].stem(samples, x_values)
    ax[0].set_title(f'{title}: Input Signal')
    ax[0].set_xlabel('n')
    ax[0].set_ylabel('x[n]')
    ax[0].grid(True)
    ax[1].stem(samples, y_values)
    ax[1].set_title(f'{title}: Output Signal')
    ax[1].set_xlabel('n')
    ax[1].set_ylabel('y[n]')
    ax[1].grid(True)
    plt.tight_layout()
    plt.show()

plot_signal_pair(x1, y1_real, N1, 'x₁[n] = sin(3πn/4)')
plot_signal_pair(x2, y2_real, N2, 'x₂[n] = cos(πn/4) + 2cos(πn/2)')